# Katbook Video Intelligence Platform — Kaggle Runner

This notebook is a **thin runner**. All the real logic lives in the importable
`katbook_vip` package (cloned from GitHub below), so you edit/test code in VS Code
and just *run* it here on a free **T4 GPU**.

**What it does:** pick video(s) → detect voice vs silent → transcribe (any language) →
sample frames adaptively → CLIP scenes + gated YOLO/OCR + BLIP-2 captions → segment →
tag each segment with Qwen (topic / subject / grade / difficulty / tags / summary) →
store in Postgres + write `results/*.json`. **One model is in VRAM at a time.**

**Before running:** Settings → Accelerator → **GPU T4 x2**, and add Secrets
`DATABASE_URL` (required to store) and `HF_TOKEN` (optional).


## Cell 0 — Configuration (the only cell you usually edit)


In [ ]:
# These keys OVERRIDE katbook_vip/config.py BASE+PROFILE. Leave a key out to keep its default.
CONFIG = {
    # SPEED PROFILE: "fast" (free T4, ~2-3 min/video) | "balanced" | "quality"
    "PROFILE": "fast",

    # WHICH videos to process from everything found under /kaggle/input:
    #   "all" | "first" | 3 (the 3rd in the printed list) | "pendulum" (name contains)
    "PROCESS": "all",

    # Re-running never redoes a video already in Postgres. Set False to FORCE redo
    # (e.g. after you changed a prompt or model).
    "SKIP_EXISTING": True,

    # On-screen OCR scripts for SILENT videos. "en" is most reliable; add "ta"/"hi"
    # to read Tamil/Hindi on-screen text (falls back to en if those weights fail).
    # Spoken transcript (Whisper) is fully multilingual regardless of this.
    "OCR_LANGS": ["en"],

    # Voice-vs-silent thresholds (validated: silent ~ -99 dB, narrated ~ -20..-30 dB).
    "SILENCE_DB": -50,
    "MIN_SPEECH_SEC": 3,
}

# Where to get the package from (public repo -> no auth needed on Kaggle):
REPO_URL = "https://github.com/MeeraVelu/Katbook_VIP_2.git"
REPO_BRANCH = "Meera"   # change if your package lives on another branch

## Cell 1 — Install dependencies (smart: skips what's already present)


In [ ]:
# Only install a package if it is MISSING or version-mismatched. Saves time/disk on
# re-runs. Kaggle's torch is built for a specific numpy; we capture that version first
# and restore it after installing so `import torch` keeps working with NO kernel restart.
import subprocess, sys, re as _re
import importlib.metadata as _md
from importlib.metadata import version as _ver, PackageNotFoundError

try:
    _BASE_NUMPY = _ver("numpy")
except Exception:
    _BASE_NUMPY = None

def _check(spec):
    name = _re.split(r"[<>=!]", spec, 1)[0].strip()
    want = spec.split("==", 1)[1] if "==" in spec else None
    try:
        have = _ver(name)
        if want is None or have == want:
            return ("ok", name, have)
        return ("update", name, spec)
    except PackageNotFoundError:
        return ("install", name, spec)

# Exactly what the katbook_vip package needs (ffmpeg/ffprobe ship with the Kaggle image;
# torch/transformers/PIL/numpy are preinstalled and get skipped automatically).
PKGS = [
    "faster-whisper==1.0.3",
    "ultralytics",                       # UNPINNED: old 8.3.0 forced numpy<2 and broke torch
    "easyocr==1.7.2",
    "keybert", "sentence-transformers==4.1.0",   # st pin: v5 breaks KeyBERT
    "kneed", "librosa", "langdetect",
    "bitsandbytes>=0.43", "accelerate>=0.30",     # 4-bit Qwen
    "psycopg2-binary", "sqlalchemy>=2.0", "pgvector",
]
_installed = [_check(p) for p in PKGS]
_to_install = [r[2] for r in _installed if r[0] != "ok"]
if _to_install:
    print(f"Installing {len(_to_install)} package(s) in one pip call; "
          f"{len(PKGS) - len(_to_install)} already present (skipped)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_to_install], check=False)
else:
    print("All dependencies already present — skipping install.")

# Restore the exact base numpy if a dep changed it (keeps torch importable, no restart).
_did = [r for r in _installed if r[0] != "ok"]
if _BASE_NUMPY and _md.version("numpy") != _BASE_NUMPY:
    print(f"numpy changed -> restoring base {_BASE_NUMPY} for torch.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"numpy=={_BASE_NUMPY}", "--no-deps"], check=False)

# spaCy English model — only download if not already loadable.
try:
    import spacy; spacy.load("en_core_web_sm"); print("spaCy en_core_web_sm: present")
except Exception:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)
    print("spaCy en_core_web_sm: downloaded")

print(f"Dependencies ready. numpy=={_md.version('numpy')} (torch-compatible). Run straight on.")

## Cell 2 — Get the `katbook_vip` package

Clones your public repo and installs it editable. If the repo is already present
(re-run), it just pulls the latest commit. Falls back to a Kaggle **Dataset** copy
if git is unavailable.


In [ ]:
import os, sys, subprocess
from pathlib import Path

PKG_PARENT = Path("/kaggle/working/Katbook_VIP_2")

def _sh(cmd, cwd=None):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=False)

if (PKG_PARENT / ".git").exists():
    _sh(["git", "-C", str(PKG_PARENT), "fetch", "--quiet", "origin", REPO_BRANCH])
    _sh(["git", "-C", str(PKG_PARENT), "reset", "--hard", f"origin/{REPO_BRANCH}"])
else:
    r = _sh(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PKG_PARENT)])
    if r.returncode != 0:
        # Fallback: a Kaggle Dataset named like the repo (add it via "+ Add Input").
        cands = list(Path("/kaggle/input").glob("**/katbook_vip/__init__.py"))
        if cands:
            PKG_PARENT = cands[0].parent.parent
            print("Using package from Kaggle input:", PKG_PARENT)
        else:
            raise RuntimeError("Could not clone repo and no katbook_vip dataset was attached.")

# Make the package importable for THIS kernel (editable install is best-effort).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PKG_PARENT)], check=False)
if str(PKG_PARENT) not in sys.path:
    sys.path.insert(0, str(PKG_PARENT))

import katbook_vip
print("katbook_vip version:", katbook_vip.__version__, "from", katbook_vip.__file__)

## Cell 3 — Secrets (DATABASE_URL required to store; HF_TOKEN optional)


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
except Exception:
    sec = None

def _secret(name):
    if sec is None:
        return os.environ.get(name)
    try:
        return sec.get_secret(name)
    except Exception:
        return None

DATABASE_URL = _secret("DATABASE_URL")
if DATABASE_URL:
    os.environ["DATABASE_URL"] = DATABASE_URL
    print("DATABASE_URL loaded — results will be stored in Postgres.")
else:
    print("WARNING: no DATABASE_URL secret. Pipeline runs JSON-only (results/ folder).")

HF_TOKEN = _secret("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded.")
else:
    print("No HF_TOKEN (fine — Whisper/CLIP/YOLO/BLIP-2/Qwen used here are not gated).")

## Cell 4 — Run the pipeline

Discovers every `.mp4` under `/kaggle/input`, prints a numbered list, processes the
ones `CONFIG["PROCESS"]` selects, and stores each to Postgres + `results/`. Per-video
errors are isolated (one bad file won't stop the batch). Safe to re-run — finished
videos are skipped.


In [ ]:
from katbook_vip import load_config, run_batch

cfg = load_config(CONFIG)
cfg["DATABASE_URL"] = os.environ.get("DATABASE_URL")   # from the secrets cell above
summary = run_batch(cfg)
summary

## Cell 5 — Peek at the results written this run

The durable copy is in Postgres; pull it to your laptop with
`python sync_results.py --watch`. Here we just print what landed in
`/kaggle/working/results/` so you can eyeball the tags.


In [ ]:
import json, glob
files = sorted(glob.glob("/kaggle/working/results/*.json"))
files = [f for f in files if not f.endswith("all_results.json")]
print(f"{len(files)} result file(s):\n")
for f in files:
    r = json.load(open(f))
    speech = "voice" if r.get("has_speech") else "SILENT"
    print(f"=== {r['source'].split('/')[-1]}  [{speech}, {r.get('language')}, "
          f"{r['segment_count']} segs, {r.get('pipeline_time_sec')}s] ===")
    for s in r["segments"]:
        tags = ", ".join(s.get("tags", [])[:5])
        print(f"  [{s['start']:>6.1f}-{s['end']:>6.1f}] {s.get('subject')} / {s.get('topic')}"
              f"  (grade {s.get('grade')}, {s.get('difficulty')}, conf {s.get('confidence')})")
        if tags:
            print(f"            tags: {tags}")
    print()

## Cell 6 — Optional: search UI (Gradio) over the stored segments

A tiny semantic + keyword search across everything you've processed. Skip it if you
only need the JSON. Uses the same Postgres, no GPU needed.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.0"], check=False)
import gradio as gr
from sqlalchemy import create_engine, text as sql
from sentence_transformers import SentenceTransformer

_eng = create_engine(os.environ["DATABASE_URL"], pool_pre_ping=True) if os.environ.get("DATABASE_URL") else None
_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

def _search(query, subject, k):
    if _eng is None:
        return "No DATABASE_URL — nothing to search."
    qv = str(_embedder.encode([query], normalize_embeddings=True)[0].tolist())
    where, params = "", {"qv": qv, "k": int(k)}
    if subject and subject != "Any":
        where = "WHERE s.llm->>'subject' = :subj"; params["subj"] = subject
    stmt = sql(f"""SELECT s.start_sec, s.end_sec, s.llm->>'topic' topic,
        s.llm->>'subject' subject, v.source_path,
        1-(s.embedding <=> :qv) score
        FROM segments s JOIN videos v ON v.video_id=s.video_id
        {where} ORDER BY s.embedding <=> :qv LIMIT :k""")
    with _eng.connect() as cx:
        rows = cx.execute(stmt, params).fetchall()
    if not rows:
        return "No matches."
    out = []
    for r in rows:
        name = (r.source_path or "").split("/")[-1]
        ts = f"{int(r.start_sec//60)}:{int(r.start_sec%60):02d}"
        out.append(f"**{r.topic}**  ({r.subject})  score={r.score:.3f}\n"
                   f"  {name} @ {ts}")
    return "\n\n".join(out)

subjects = ["Any"]
if _eng is not None:
    with _eng.connect() as cx:
        subjects += [r[0] for r in cx.execute(sql(
            "SELECT DISTINCT llm->>'subject' FROM segments "
            "WHERE llm->>'subject' IS NOT NULL ORDER BY 1")).fetchall()]

demo = gr.Interface(
    fn=_search,
    inputs=[gr.Textbox(label="Search", value="time period of a pendulum"),
            gr.Dropdown(subjects, value="Any", label="Subject"),
            gr.Slider(1, 10, value=5, step=1, label="Results")],
    outputs=gr.Markdown(),
    title="Katbook VIP — segment search",
    description="Semantic search over processed video segments (pgvector).")
demo.launch(share=True)